# Hospital Transfer Acceptance and Ranking

Build decision-level records, compare four model families, tune XGBoost with encounter-grouped cross-validation, and rank candidate hospitals. The bundled data is synthetic; generated scores illustrate the workflow.


In [ ]:
from pathlib import Path
import sys

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "generate_synthetic_data.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Run from the repository root or notebooks directory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))


import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit, GroupKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from xgboost import XGBClassifier

import json
from transfer_ranking import FEATURES, build_decisions, topk_capture, lift_over_random


## 1. Load synthetic data


In [ ]:
DATA_DIR = ROOT / "data" / "synthetic"

df_all = pd.read_csv(DATA_DIR / "synthetic_all_disch.csv")
df_ctal = pd.read_csv(DATA_DIR / "synthetic_ctal_disch.csv")
df_ctal = df_ctal.rename(columns={"csn": "CSN"})

print("Encounter rows:", len(df_all))
print("Transfer rows:", len(df_ctal))


## 2. Build hospital-level decision records


In [ ]:
df_dec = build_decisions(df_all, df_ctal)
print("Decision-level rows:", len(df_dec))
print("Matched encounters:", df_dec["CSN"].nunique())
display(df_dec[["CSN", "Hospital", "choice_rank", "Status", "y_accept"]].head())


## 3. Features and decision timing

Context is joined on both encounter and patient identifiers. Unknown statuses are excluded, and ambiguous or missing joins are rejected.

This is a retrospective analysis: final length of stay, discharge service, and discharge timestamps may be unavailable at outreach time. An operational model would need their as-of-decision equivalents. Outcome and destination fields are excluded from predictors.


In [ ]:
display(df_dec[FEATURES].head(3))
assert not {"PAT_ID", "CSN", "Status", "y_accept", "Patient_Transferred_to_Site"}.intersection(FEATURES)


## 4. Encounter-grouped holdout and cross-validation

Reserve 20% of encounters for the test set. Use three grouped folds within the remaining 80% for model comparison and tuning. Every preprocessing pipeline is fitted inside its training fold.


In [ ]:
X = df_dec[FEATURES].copy()
y = df_dec["y_accept"].copy()
groups = df_dec["CSN"].copy()

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx].copy(), X.iloc[test_idx].copy()
y_train, y_test = y.iloc[train_idx].copy(), y.iloc[test_idx].copy()

assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[test_idx]))
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

groups_train = groups.iloc[train_idx]
cv = GroupKFold(n_splits=3)
cv_splits = list(cv.split(X_train, y_train, groups_train))
for fit_idx, valid_idx in cv_splits:
    assert set(groups_train.iloc[fit_idx]).isdisjoint(set(groups_train.iloc[valid_idx]))
    assert y_train.iloc[fit_idx].nunique() == 2 and y_train.iloc[valid_idx].nunique() == 2
print("Three grouped CV folds; test encounters stay untouched during model selection.")


## 5. Preprocessing and models


In [ ]:
cat_cols = [
    c for c in X_train.columns
    if X_train[c].dtype == "object" or str(X_train[c].dtype).startswith("category")
]
num_cols = [c for c in X_train.columns if c not in cat_cols]

def make_preprocess(*, dense=False, scale_numeric=True):
    num_steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale_numeric:
        num_steps.append(("scaler", StandardScaler()))

    return ColumnTransformer(
        transformers=[
            ("num", Pipeline(num_steps), num_cols),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    (
                        "onehot",
                        OneHotEncoder(
                            handle_unknown="ignore",
                            sparse_output=not dense,
                        ),
                    ),
                ]),
                cat_cols,
            ),
        ],
        remainder="drop",
        sparse_threshold=0.0 if dense else 0.3,
    )

models = {
    "Logistic Regression": Pipeline([
        ("prep", make_preprocess(scale_numeric=True)),
        ("model", LogisticRegression(max_iter=2000, solver="liblinear")),
    ]),
    "Hist Gradient Boosting": Pipeline([
        ("prep", make_preprocess(dense=True, scale_numeric=False)),
        ("model", HistGradientBoostingClassifier(
            max_depth=6,
            learning_rate=0.1,
            max_iter=250,
            random_state=42,
        )),
    ]),
    "Random Forest": Pipeline([
        ("prep", make_preprocess(scale_numeric=False)),
        ("model", RandomForestClassifier(
            n_estimators=400,
            max_depth=12,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=42,
            n_jobs=1,
        )),
    ]),
    "XGBoost": Pipeline([
        ("prep", make_preprocess(scale_numeric=False)),
        ("model", XGBClassifier(
            n_estimators=300,
            max_depth=5,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="logloss",
            random_state=42,
            n_jobs=1,
        )),
    ]),
}


## 6. Tune and select using training folds

Compare logistic regression, histogram gradient boosting, random forest, and XGBoost. Search eight XGBoost settings. Select the model with the lowest mean cross-validation Brier score before evaluating the test set.


In [ ]:
cv_rows = []
for name, model in models.items():
    scores_cv = -cross_val_score(model, X_train, y_train, cv=cv_splits,
                                scoring="neg_brier_score", n_jobs=1)
    cv_rows.append({"model": name, "cv_brier": scores_cv.mean()})

search = GridSearchCV(
    models["XGBoost"],
    {"model__max_depth": [2, 4], "model__learning_rate": [0.03, 0.1],
     "model__n_estimators": [150, 300]},
    scoring="neg_brier_score", cv=cv_splits, refit=True, n_jobs=1,
)
search.fit(X_train, y_train)
models["XGBoost (tuned)"] = search.best_estimator_
cv_rows.append({"model": "XGBoost (tuned)", "cv_brier": -search.best_score_})
cv_results = pd.DataFrame(cv_rows).sort_values("cv_brier").reset_index(drop=True)
best_name = cv_results.iloc[0]["model"]
print("Selected using training folds:", best_name)
print("Best XGBoost parameters:", search.best_params_)
display(cv_results.round(4))

# Freeze selection before computing any test metrics.
results, scores = [], {}
for name, model in models.items():
    model.fit(X_train, y_train)
    score = model.predict_proba(X_test)[:, 1]
    scores[name] = score
    row = {"model": name, "roc_auc": roc_auc_score(y_test, score),
           "avg_precision": average_precision_score(y_test, score),
           "brier": brier_score_loss(y_test, score)}
    for fraction in [0.10, 0.20, 0.30]:
        label = f"top{int(fraction * 100)}%"
        row[f"capture@{label}"] = topk_capture(y_test, score, fraction)
        row[f"lift@{label}"] = lift_over_random(y_test, score, fraction)
    results.append(row)
df_results = pd.DataFrame(results).merge(cv_results, on="model")
df_results["selected_by_cv"] = df_results["model"].eq(best_name)
display(df_results.round(4))

OUTPUT_DIR = ROOT / "runs" / "modeling"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_results.to_csv(OUTPUT_DIR / "model_comparison.csv", index=False)
summary = {
    "data": "synthetic", "selected_model": best_name,
    "encounters": int(df_dec.CSN.nunique()), "decisions": len(df_dec),
    "train_encounters": int(groups_train.nunique()),
    "test_encounters": int(groups.iloc[test_idx].nunique()),
    "train_decisions": len(X_train), "test_decisions": len(X_test),
    "cv_folds": 3, "xgboost_candidates": len(search.cv_results_["params"]),
    "best_xgboost_params": search.best_params_,
    "models": df_results.to_dict(orient="records"),
}
(OUTPUT_DIR / "metrics.json").write_text(json.dumps(summary, indent=2) + "\n")


## 7. Test-set capture curve

Capture measures the share of observed accepts among the highest-scored decisions pooled across test encounters. It does not measure top-K hospital success within each encounter or prove a reduction in real outreach effort.


In [ ]:
best_scores = scores[best_name]

ranked = (
    pd.DataFrame({"y_true": y_test.to_numpy(), "score": best_scores})
    .sort_values("score", ascending=False, kind="stable")
    .reset_index(drop=True)
)

ranked["fraction_contacted"] = (np.arange(len(ranked)) + 1) / len(ranked)
total_accepts = ranked["y_true"].sum()
ranked["fraction_accepts_captured"] = (
    ranked["y_true"].cumsum() / total_accepts
)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(
    np.r_[0, ranked["fraction_contacted"]],
    np.r_[0, ranked["fraction_accepts_captured"]],
    label=best_name,
)
ax.plot([0, 1], [0, 1], linestyle="--", label="Random ordering")
ax.set_title("Synthetic Cumulative Capture Curve")
ax.set_xlabel("Fraction of candidate decisions contacted")
ax.set_ylabel("Fraction of accepts captured")
ax.legend()
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "capture_curve.svg", bbox_inches="tight")
plt.show()


## 8. Rank candidate hospitals for one synthetic encounter


In [ ]:
best_model = models[best_name]

demo_row = df_dec.iloc[test_idx[0]]
case_features = {
    "choice_rank": 1,
    "Type_of_Transfer": demo_row["Type_of_Transfer"],
    "age": demo_row["age"],
    "Sex": demo_row["Sex"],
    "LOS_cat": demo_row["LOS_cat"],
    "discharge_Service": demo_row["discharge_Service"],
    "hRegion": demo_row["hRegion"],
    "LHIN_Name": demo_row["LHIN_Name"],
    "discharge_hour": demo_row["discharge_hour"],
    "discharge_dow": demo_row["discharge_dow"],
    "discharge_month": demo_row["discharge_month"],
}

candidate_hospitals = sorted(X_train["Hospital"].dropna().unique())
rows = []

for hospital in candidate_hospitals:
    row = case_features.copy()
    row["Hospital"] = hospital
    rows.append(row)

X_case = pd.DataFrame(rows)[FEATURES]
probabilities = best_model.predict_proba(X_case)[:, 1]

ranking = (
    pd.DataFrame({
        "Hospital": candidate_hospitals,
        "Predicted Acceptance Probability": probabilities,
    })
    .sort_values("Predicted Acceptance Probability", ascending=False)
    .reset_index(drop=True)
)

display(ranking.head(8).assign(**{
    "Predicted Acceptance Probability": ranking["Predicted Acceptance Probability"].map("{:.1%}".format)
}))

ranking.to_csv(OUTPUT_DIR / "candidate_ranking.csv", index=False)
assert len(ranking) == len(candidate_hospitals)
assert ranking["Predicted Acceptance Probability"].between(0, 1).all()


## Interpretation

The workflow links encounters to candidate-hospital decisions, tunes models without using test encounters, and reports probability quality and pooled ranking metrics. Candidate rankings hold context and choice rank constant while varying the hospital. Predictions for hospitals not historically contacted are estimates with unobserved outcomes.

All outputs in this notebook come from synthetic data. See [evaluation methodology](../docs/evaluation.md) for metric definitions and the project results summarized on the resume.
